# Pipeline mimiSBM générique

- Le chargement des matrices est piloté par un simple dict `{nom_vue: chemin}` défini en configuration
- Cellules `n_init` répétées et 3 versions successives de la recherche de grille K×Q sont factorisés en deux fonctions réutilisables : `fit_best_of_n_init` et `grid_search`.
- **Test de stabilité** : pour le couple (K, Q) retenu, on refait plusieurs runs
   et on mesure l'ARI (Adjusted Rand Index) entre les partitions obtenues, pour savoir si le
   clustering est reproductible ou sensible à l'initialisation
- **Test de validation sur données simulées** : un petit tenseur est généré à
   partir de K, Q connus, et on vérifie que la recherche de grille retrouve bien ces valeurs.
   Ça sert de test unitaire du pipeline, indépendant de la disponibilité des vraies données.

## Imports

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

from sklearn.metrics import adjusted_rand_score

from mimisbm_main.mimisbm._base import MimiSBM

## 1. Configuration — chemins des matrices d'adjacence et hyperparametres

Le dict `ADJACENCY_PATHS` associe un nom de vue (libre) a un chemin de fichier CSV.

In [ ]:
# nom de vue -> chemin matrice d'adjacence correspondante
ADJACENCY_PATHS = {
    "vue_1": Path(r"/mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Données/adjacency/lda/adjacency_matrix_silh_Ka.csv"),
    "vue_2": Path(r"/mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Données/adjacency/lda/adjacency_matrix_silh_PPI_Jaccard.csv"),
    "vue_3": Path(r"/mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Données/adjacency/lda/adjacency_matrix_silh_TE_Coverage_Asymmetry.csv"),
    "vue_4": Path(r"/mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Données/adjacency/lda/adjacency_matrix_silh_TE_Number_Asymmetry.csv"),
    "vue_5": Path(r"/mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Données/adjacency/lda/adjacency_matrix_silh_TE_Density_Asymmetry.csv"),
}

OUTPUT_DIR = Path(r"/mnt/c/Users/thoma/OneDrive/Documents/ENSIIE/Master/Stage 3A/Plots/mimiSBM")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparametres de la recherche de modele
K_VALUES = range(2, 11)   # nombre de clusters d'individus a tester
Q_VALUES = range(1, 6)    # nombre de composantes de vues a tester
N_INIT = 10                # nombre d'initialisations par couple (K, Q), equivalent au n_init R
PRIORS = "jeffreys"        # ou "uniform", ou un tableau numpy
MAX_ITER = 100
TOL = 1e-4
BASE_SEED = 0

## 2. Chargement generique et validation des matrices d'adjacence

In [ ]:
def load_adjacency_matrices(paths: dict) -> dict:
    """Charge et valide un ensemble de matrices d'adjacence binaires.

    Args:
        paths: dict {nom_de_vue: chemin_matrice_d'adjacence}.

    Returns:
        dict {nom_de_vue: matrice numpy (N x N) binaire}.

    Raises:
        ValueError: si une matrice contient des valeurs hors {0, 1}, ou si les tailles
            des matrices ne sont pas toutes identiques.
    """
    adj_dict = {}
    shapes = set()
    for name, path in paths.items():
        path = Path(path)
        adj = pd.read_csv(path, index_col=0).values

        uniq = np.unique(adj)
        print(f"[{name}] shape={adj.shape}, valeurs uniques={sorted(uniq.tolist())}")
        if not np.all(np.isin(uniq, [0, 1])):
            raise ValueError(f"{name} : valeurs differentes de 0 et 1 detectees.")

        adj_dict[name] = adj.astype(np.int64)
        shapes.add(adj.shape[0])

    if len(shapes) > 1:
        raise ValueError(f"Tailles de matrices incoherentes entre vues : {shapes}")

    return adj_dict

## 3. Assemblage et verification du tenseur A (N x N x V)

Meme logique que le notebook original (diagonale mise a 0, symetrisation de securite,
assertions de forme/symetrie/diagonale), extraite en fonction reutilisable.

In [ ]:
def build_tensor(adj_dict: dict):
    """Assemble les matrices d'adjacence des vues en tenseur (N, N, V).

    Returns:
        (A, names) ou A est le tenseur numpy et names l'ordre des vues dans le tenseur.
    """
    names = list(adj_dict.keys())
    N = adj_dict[names[0]].shape[0]
    V = len(names)

    assert all(adj_dict[n].shape == (N, N) for n in names), "Toutes les vues doivent etre NxN de meme taille."

    A = np.zeros((N, N, V), dtype=np.int64)
    for i, name in enumerate(names):
        adj = adj_dict[name].copy()
        np.fill_diagonal(adj, 0)
        if not np.array_equal(adj, adj.T):
            adj = np.maximum(adj, adj.T)
        A[:, :, i] = adj

    assert A.shape == (N, N, V)
    assert all(np.array_equal(A[:, :, v], A[:, :, v].T) for v in range(V))
    assert all(np.all(np.diag(A[:, :, v]) == 0) for v in range(V))

    return A, names

## 4. Chargement effectif a partir de la configuration

In [ ]:
adj_dict = load_adjacency_matrices(ADJACENCY_PATHS)
A, view_names = build_tensor(adj_dict)

N, _, V = A.shape
print(f"\nTenseur A : shape={A.shape}  (N={N} individus, V={V} vues)")

tensor_path = OUTPUT_DIR / "adjacency_tensor.npy"
np.save(tensor_path, A)
print(f"Tenseur sauvegarde : {tensor_path}")

## 5. Ajustement mimiSBM multi-init (equivalent `n_init` R)

Une seule fonction : on relance `fit()` `n_init` fois avec des graines differentes et on garde le run au
meilleur ELBO, exactement le role que jouait `n_init=10` cote R.

In [ ]:
def fit_best_of_n_init(A, K, Q, n_init=N_INIT, priors=PRIORS, max_iter=MAX_ITER,
                        tol=TOL, base_seed=BASE_SEED, verbose=False):
    """Ajuste MimiSBM plusieurs fois pour (K, Q) et retient le meilleur ELBO.

    Returns:
        (best_model, best_seed)
    """
    best_model, best_elbo, best_seed = None, -np.inf, None
    for i in range(n_init):
        seed = base_seed + i
        model = MimiSBM(
            n_clusters=K, n_components=Q,
            clusters_prior=priors, components_prior=priors, adjacency_prior=priors,
            max_iter=max_iter, tol=tol, random_state=seed,
        )
        model.fit(A)
        if verbose:
            print(f"  run {i} (seed={seed}) : ELBO={model.elbo_:.2f}  converged={model.converged_}")
        if model.elbo_ > best_elbo:
            best_elbo, best_model, best_seed = model.elbo_, model, seed
    return best_model, best_seed

## 6. Recherche de grille (K, Q) — selection de modele par ELBO

Une seule implementation propre de la boucle K x Q x n_init

In [ ]:
def grid_search(A, K_values, Q_values, n_init=N_INIT, **kwargs):
    """Recherche de grille (K, Q) : garde le meilleur modele par couple, et le meilleur global.

    Returns:
        (results_df, best) ou best = {"elbo", "model", "K", "Q", "seed"}.
    """
    results = []
    best = {"elbo": -np.inf, "model": None, "K": None, "Q": None, "seed": None}
    for K in K_values:
        for Q in Q_values:
            model, seed = fit_best_of_n_init(A, K, Q, n_init=n_init, **kwargs)
            results.append({
                "K": K, "Q": Q, "ELBO": model.elbo_,
                "converged": model.converged_, "best_seed": seed,
            })
            print(f"K={K}, Q={Q} : meilleur ELBO = {model.elbo_:.2f}")
            if model.elbo_ > best["elbo"]:
                best.update(elbo=model.elbo_, model=model, K=K, Q=Q, seed=seed)
    return pd.DataFrame(results), best


results_df, best = grid_search(A, K_VALUES, Q_VALUES, n_init=N_INIT)

print()
print(f"Meilleur couple (K, Q) = ({best['K']}, {best['Q']})  |  ELBO = {best['elbo']:.2f}")

## 7. Visualisation de la grille ELBO(K, Q)

In [ ]:
elbo_matrix = results_df.pivot(index="K", columns="Q", values="ELBO")

plt.figure(figsize=(9, 6))
plt.imshow(elbo_matrix.values, aspect="auto", cmap="viridis")
plt.xticks(range(len(elbo_matrix.columns)), elbo_matrix.columns)
plt.yticks(range(len(elbo_matrix.index)), elbo_matrix.index)
plt.xlabel("Nombre de composantes de vues Q")
plt.ylabel("Nombre de clusters K")
plt.title("ELBO maximal pour chaque couple (K, Q)")
for i in range(elbo_matrix.shape[0]):
    for j in range(elbo_matrix.shape[1]):
        plt.text(j, i, f"{elbo_matrix.iloc[i, j]:.1f}", ha="center", va="center",
                  color="white", fontsize=8)
plt.colorbar(label="ELBO")
plt.tight_layout()
plt.show()

results_df.sort_values("ELBO", ascending=False).reset_index(drop=True)

## 8. Resultats du meilleur modele

In [ ]:
best_model = best["model"]
node_clusters, view_components = best_model.predict()

print("Tailles des clusters d'individus (K):")
print(pd.Series(node_clusters).value_counts().sort_index())

print()
print("Composante de vue retenue par vue:")
print(pd.DataFrame({"vue": view_names, "composante": view_components}))

model_path = OUTPUT_DIR / "mimiSBM_best_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(best_model, f)
print(f"\nModele sauvegarde : {model_path}")

## 9. Test de stabilite du clustering entre initialisations (ARI)

Pour le couple (K, Q) retenu, on relance plusieurs fits independants et on mesure l'Adjusted Rand Index (ARI) entre chaque paire de partitions obtenues : un ARI proche de 1 signifie que le clustering est
stable (peu sensible a l'initialisation), un ARI faible signale un modele instable pour ce
(K, Q) — auquel cas la valeur d'ELBO seule ne suffit pas a faire confiance au resultat.

In [ ]:
def stability_test(A, K, Q, n_init=N_INIT, **kwargs):
    """Mesure la stabilite (ARI) du clustering entre plusieurs runs pour un (K, Q) fixe."""
    zs, ws = [], []
    for i in range(n_init):
        model, _ = fit_best_of_n_init(A, K, Q, n_init=1, base_seed=1000 + i, **kwargs)
        z, w = model.predict()
        zs.append(z); ws.append(w)
    ari_z = [adjusted_rand_score(a, b) for a, b in combinations(zs, 2)]
    ari_w = [adjusted_rand_score(a, b) for a, b in combinations(ws, 2)]
    return ari_z, ari_w


ari_z, ari_w = stability_test(A, best["K"], best["Q"], n_init=N_INIT)

print(f"ARI clusters individus (K={best['K']})  - moyenne={np.mean(ari_z):.3f}, min={np.min(ari_z):.3f}")
print(f"ARI composantes de vues (Q={best['Q']}) - moyenne={np.mean(ari_w):.3f}, min={np.min(ari_w):.3f}")

## 10. Test de validation — pipeline sur donnees simulees (K, Q connus)

Test unitaire independant des vraies donnees : on simule un tenseur multicouche a partir
d'un K et d'un Q connus (modele generatif mimiSBM : blocs de connectivite Bernoulli par
cluster/composante), puis on verifie que `grid_search` retrouve bien ces valeurs et que le
clustering obtenu correspond a la verite (ARI).

In [ ]:
def simulate_mimisbm_tensor(N, V, K, Q, seed=0):
    """Simule un tenseur (N, N, V) suivant le modele generatif mimiSBM, K et Q connus."""
    rng = np.random.default_rng(seed)
    z_true = rng.integers(0, K, size=N)
    w_true = np.array([i % Q for i in range(V)])
    rng.shuffle(w_true)

    alpha = np.zeros((K, K, Q))
    alpha[:, :, 0] = 0.05
    np.fill_diagonal(alpha[:, :, 0], 0.9)          # composante 0 : structure assortative forte
    for s in range(1, Q):
        alpha[:, :, s] = 0.55
        np.fill_diagonal(alpha[:, :, s], 0.1)      # composantes suivantes : structure differente

    A_sim = np.zeros((N, N, V), dtype=np.int64)
    for v in range(V):
        s = w_true[v]
        P = alpha[z_true][:, z_true, s]
        draw = (rng.random((N, N)) < P).astype(np.int64)
        draw = np.triu(draw, 1)
        A_sim[:, :, v] = draw + draw.T

    return A_sim, z_true, w_true


K_TRUE, Q_TRUE = 3, 2
A_sim, z_true, w_true = simulate_mimisbm_tensor(N=60, V=6, K=K_TRUE, Q=Q_TRUE, seed=0)

sim_results_df, sim_best = grid_search(
    A_sim, K_values=range(2, 5), Q_values=range(1, 4), n_init=3, max_iter=100,
)

print()
print(f"(K, Q) retrouve par la grille : ({sim_best['K']}, {sim_best['Q']})  |  vrai (K, Q) = ({K_TRUE}, {Q_TRUE})")

z_pred, w_pred = sim_best["model"].predict()
print(f"ARI clusters individus vs verite : {adjusted_rand_score(z_true, z_pred):.3f}")
print(f"ARI composantes vues vs verite   : {adjusted_rand_score(w_true, w_pred):.3f}")

assert (sim_best["K"], sim_best["Q"]) == (K_TRUE, Q_TRUE), "Le pipeline n'a pas retrouve le (K, Q) simule."
print("\nTest de validation du pipeline : OK")